In [70]:
import pandas as pd

df = pd.read_csv("../data/mobile_money_statements.csv")
print(df.shape)
df.head(30)

(183600, 18)


,txn_id,msisdn,reg_id,account_name,region,segment,txn_time,txn_type,amount,gps_lat,gps_lon,agent_id,device_id,counterparty,balance_after,manual_review_score,settlement_status,is_fraud
0,TXN0102856,254700004022,NID101089,Joseph Mwangi,Kampala,retail,2026-06-30 14:24:19,send,"1,329/- Dr",NaN,NaN,AG1014,DV978723,CP6324,8123.0,0.061,settled,0
1,TXN0155490,254700013612,NID103658,Rehema Ndlovu,Mombasa,retail,"Jan 29, 2026 01:17 PM",send,289/- Dr,1.77470,34.70915,AG1084,DV616667,CP2831,6322.0,0.000,settled,0
2,TXN0019434,254700005862,nid101584,Mary Mutua,Arusha,retail,"Sep 16, 2025 12:55 AM",receive,"1,571/-",-3.74336,33.67018,AG1091,DV637734,CP167,15237.0,0.912,reversed,1
3,TXN0049426,254700009223,NID102493,Joseph Okoth,Dar es Salaam,retail,07/04/2026 11:21,send,599/- Dr,NaN,NaN,AG1047,DV666470,CP3382,3322.0,0.095,settled,0
4,TXN0013816,254700014760,nid103963,B. Ssentongo,Mwanza,retail,20/12/2025 11:05,cashout,"(1,307/-)",0.08284,29.02835,NaN,NaN,CP8428,2168.0,0.085,settled,0
5,TXN0068766,254700005080,NID101375,Grace Mutua,Arusha,retail,2026-06-21 12:00:08,send,"2,128/- Dr",-0.28486,33.04071,NaN,DV322595,CP6317,3202.0,0.066,settled,0
6,TXN0006514,254700010321,NID102782,SSENTONGO GRACE,Mwanza,retail,24/07/2026 04:01,bill,"-2,136/-",-3.07853,33.77321,NaN,DV492573,CP7821,4046.0,0.008,settled,0
7,TXN0115001,254700007185,NID101945,Amina Ali,Mombasa,retail,"May 25, 2026 12:17 PM",cashin,"1,026/-",NaN,NaN,AG1026,DV104452,CP6532,3226.0,0.172,settled,0
8,TXN0067557,254700011799,nid103172,G. Kamau,Mombasa,merchant,22/10/2025 22:44,cashin,"1,099/-",2.62258,33.90121,AG1099,DV674871,CP5021,6475.0,0.069,settled,0
9,TXN0086664,254700005680,NID101533,Aisha Ali,Nairobi,retail,2025-09-02 07:48:24,receive,951/-,NaN,NaN,AG1338,DV107852,CP609,4576.0,0.090,pending,0


In [71]:
# Which columns have missing values and to which extend?
miss = df.isna().mean().sort_values(ascending=False) * 100
print(miss[miss > 0])

gps_lon      36.210784
gps_lat      36.210784
agent_id     13.693900
device_id     5.912854
dtype: float64


In [72]:
df.groupby('segment')['gps_lat'].apply(lambda x: x.isna().mean() * 100)

segment
agent       35.602315
merchant    36.207089
retail      36.250747
Name: gps_lat, dtype: float64

In [73]:
# percentage of missing agent-id per txn_type
df.groupby('txn_type')['agent_id'].apply(lambda x: x.isna().mean() * 100)

txn_type
airtime    14.089273
bill       13.741711
cashin     13.558851
cashout    13.603029
paybill    13.543275
receive    13.880663
send       13.516643
Name: agent_id, dtype: float64

In [74]:
# Missing percent of agent-id according to the region
df.groupby('region')['agent_id'].apply(lambda x: x.isna().mean() * 100)

region
Arusha           45.228055
Dar es Salaam     0.000000
Eldoret           0.000000
Jinja            45.050015
Kampala           0.000000
Kigali            0.000000
Mombasa           0.000000
Mwanza           44.780816
Nairobi           0.000000
Nakuru            0.000000
Name: agent_id, dtype: float64

In [75]:
#Missing device-id according to the customer-id
df.groupby('msisdn')['device_id'].apply(lambda x: x.isna().mean() * 100).describe()

count    4825.000000
mean        5.854062
std         5.882479
min         0.000000
25%         0.000000
50%         5.164319
75%         8.333333
max       100.000000
Name: device_id, dtype: float64

In [76]:
# Duplicates
print("Exact duplicate rows:", df.duplicated().sum())
print("Duplicate txn_id:", df['txn_id'].duplicated().sum())

# Consistency — amount format
print("\nSample amount values:")
print(df['amount'].sample(10, random_state=1).tolist())

# Consistency — txn_time format
print("\nSample txn_time values:")
print(df['txn_time'].sample(10, random_state=1).tolist())

# Consistency — region spelling
print("\nUnique region values:", df['region'].unique())

# Validity — impossible amounts
print("\nAmount describe (as string, so this may error — that's itself evidence!):")
try:
    print(df['amount'].astype(float).describe())
except Exception as e:
    print("Could not convert directly:", e)

Exact duplicate rows: 3600
Duplicate txn_id: 3600

Sample amount values:
['71/- Dr', '4,742/- Dr', '1,701/-', '(227/-)', '(165/-)', '(51/-)', '-316/-', '-2,130/-', '638/-', '-62/-']

Sample txn_time values:
['2026-05-06 02:37:02', '27/02/2025 13:26', '2025-10-13 02:15:03', '2026-01-07 21:25:57', '2025-09-19 10:38:43', '26/05/2025 11:06', '2026-01-12 13:59:34', 'May 06, 2026 11:08 AM', '19/03/2026 22:04', '30/03/2025 13:59']

Unique region values: <StringArray>
[      'Kampala',       'Mombasa',        'Arusha', 'Dar es Salaam',
        'Mwanza',       'Nairobi',       'Eldoret',        'Kigali',
        'Nakuru',         'Jinja']
Length: 10, dtype: str

Amount describe (as string, so this may error — that's itself evidence!):
Could not convert directly: could not convert string to float: '1,329/- Dr'


In [77]:
#Does one number identify a unique customer?
print("Unique msisdn:", df['msisdn'].nunique())
print("Unique reg_id:", df['reg_id'].nunique())
print("Unique account_name:", df['account_name'].nunique())

Unique msisdn: 4825
Unique reg_id: 11722
Unique account_name: 1232


In [78]:
miss_rate = df.groupby('msisdn')['device_id'].apply(lambda x: x.isna().mean() * 100)
txn_count = df.groupby('msisdn').size()

comparison = pd.DataFrame({'missing_pct': miss_rate, 'num_transactions': txn_count})
comparison.sort_values('missing_pct', ascending=False).head(10)

,missing_pct,num_transactions
msisdn,,
254700011213,100.000000,1
254700002041,66.666667,3
254700003593,50.000000,2
254700010726,50.000000,8
254700010050,50.000000,2
254700012383,50.000000,4
254700002383,50.000000,2
254700008364,50.000000,4
254700001654,50.000000,2


In [79]:
print(df['balance_after'].describe())
print()
print("Negative balances:", (df['balance_after'] < 0).sum())
print()
print(df['segment'].value_counts())
print()
print(df['counterparty'].nunique(), "unique counterparties")
print(df['manual_review_score'].describe())
print()
print(df['settlement_status'].value_counts())

count    183600.000000
mean       7990.912761
std        5650.070263
min          13.000000
25%        3850.750000
50%        6713.000000
75%       10754.000000
max       58933.000000
Name: balance_after, dtype: float64

Negative balances: 0

segment
retail      147208
merchant     26887
agent         9505
Name: count, dtype: int64

8999 unique counterparties
count    183600.000000
mean          0.142947
std           0.239773
min           0.000000
25%           0.040000
50%           0.076000
75%           0.115000
max           1.000000
Name: manual_review_score, dtype: float64

settlement_status
settled     157856
reversed     10101
pending       8404
held          7239
Name: count, dtype: int64


## PART I:  Data Quality Audit

Before running any diagnostics, I did a first visual scan of the dataset and flagged a few 
columns that looked off just from eyeballing the first rows: `reg_id`, `account_name`, 
`txn_time`, and `amount` looked inconsistently formatted, while `gps_lat`, `gps_lon`, 
`agent_id`, and `device_id` had visible gaps. Running `df.isna().mean()` confirmed the 
missingness suspects and gave exact numbers: `gps_lon`/`gps_lat` at 36.21% missing, `agent_id` 
at 13.69%, and `device_id` at 5.91%.

For `gps_lat`/`gps_lon`, my first instinct was that missingness was tied to phone type, since 
the two columns share an identical missing rate (36.21%), suggesting they go missing together 
as a pair. The lab brief describes this dataset as having "MNAR-missing GPS" tied to 
feature-phone users, but there is no `device_type` column in this dataset to verify that 
directly. I tested the closest available proxy, `segment` (retail/merchant/agent), and found 
missingness nearly identical across all three (35.6%, 36.2%, 36.3%) meaning segment does 
not explain it either. Since I cannot identify any observed column that explains the 
missingness, and the stated design intent (per the lab brief) is that this reflects something 
about the phone itself which I cannot observe in this data. I'm therefore classifying this as MNAR: 
the missingness plausibly depends on an attribute of the customer's device that isn't 
captured anywhere in the dataset, rather than something I could verify as MAR.

For `agent_id`, I initially guessed the missingness came from certain transaction types not 
involving an agent (e.g. a phone-to-phone transfer shouldn't need one). This turned out to be 
wrong: grouping missingness by `txn_type` showed every category sitting at roughly the same 
13-14%, meaning transaction type explains nothing here. My second hypothesis was regional — 
and this one held up strongly. Grouping by `region` showed Arusha, Mwanza, and Jinja all 
sitting around 45% missing, while every other region showed exactly 0%. This is MAR: the 
missingness is explained by region, most likely because agent networks in these areas 
(outside the core national network) don't report `agent_id` back to the system.

For `device_id`, I couldn't find an explanatory column at first. I tried grouping missingness 
by phone number (`msisdn`, used as a customer proxy since there's no dedicated customer_id), 
which showed a wide, gradual spread from 0% to 100% rather than a sharp jump like `agent_id` 
had no obvious explanation. Digging further, I checked whether the customers with the 
highest missing rates simply had very few total transactions, and this was the case: every 
customer showing a high missing percentage had only 1-8 transactions total, meaning a single 
missing row for a low-activity customer inflates their personal "missing rate" without 
reflecting any real pattern. Since no observed column explains the missingness, and it appears 
statistically scattered rather than concentrated, I'm classifying `device_id` as MCAR.

While investigating identity columns, I found a real data quality issue: msisdn (phone 
number) has 4,825 unique values, account_name has only 1,232, and reg_id has 11,722 despite 
the dataset supposedly representing ~4,000 customers. This confirms the multi-SIM problem the 
textbook case study warned about: the same real person is represented inconsistently across 
these three columns, which will need entity resolution before I can aggregate to a 
customer-level feature table or split into train/test safely.

I also confirmed 3,600 exact duplicate rows (matching exactly 3,600 duplicate txn_ids), 
confirmed `amount` is stored as inconsistently formatted text (mixing commas, "Dr" suffixes, 
parentheses for negatives, and plain minus signs. e.g. "71/- Dr", "(227/-)", "-316/-"), and 
confirmed `txn_time` mixes three different date formats (ISO, DD/MM/YYYY, and worded dates 
like "May 06, 2026 11:08 AM"). I checked `region` for the same kind of inconsistent-spelling 
issue but found it clean: 10 distinct values with no variants.

Checking the remaining columns: `balance_after` is clean and has no missing values, no negative 
balances (min is 13, a plausible value), sensible distribution, no remediation needed. 
`segment` (retail/merchant/agent) and `counterparty` (8,999 unique values) both look 
structurally fine, though `counterparty`'s high cardinality means it will need frequency or 
hashing-based encoding later rather than one-hot. `manual_review_score` and 
`settlement_status` were also checked, and while neither has a missingness problem, both are 
flagged as a **timeliness** concern rather than a missingness one.

This brings my audit to 10 columns reviewed for data quality issues: `gps_lat`, `gps_lon`, 
`agent_id`, `device_id`, `amount`, `txn_time`, `msisdn`, `account_name`, `reg_id`, and 
duplicate `txn_id` rows. Two further columns, `manual_review_score` and `settlement_status` 
were flagged separately for a suspected timeliness/leakage issue, to be resolved in the 
Leakage Assessment section (Section 3).

# Part II: Data Cleaning & Feature Engineering

## Customer Identifier Decision

In [80]:
# Cleaning "amount" to have consistent values

import re

def clean_amount(val):
    val = str(val).strip()
    is_negative = 'Dr' in val or val.startswith('(') or val.startswith('-')
    num = re.sub(r'[^\d.]', '', val)  # strip everything except digits and decimal point
    num = float(num) if num else 0.0
    return -num if is_negative else num

df['amount_clean'] = df['amount'].apply(clean_amount)
df[['amount', 'amount_clean']].sample(10, random_state=1)

,amount,amount_clean
79900,71/- Dr,-71.0
57056,"4,742/- Dr",-4742.0
133219,"1,701/-",1701.0
42819,(227/-),-227.0
178284,(165/-),-165.0
87476,(51/-),-51.0
68436,-316/-,-316.0
155665,"-2,130/-",-2130.0
61723,638/-,638.0
31418,-62/-,-62.0


In [81]:
#Cleaning txn_time

def parse_txn_time(val):
    val = str(val).strip()
    if re.match(r'^\d{4}-\d{2}-\d{2}', val):
        return pd.to_datetime(val, format='%Y-%m-%d %H:%M:%S')
    elif re.match(r'^\d{2}/\d{2}/\d{4}', val):
        return pd.to_datetime(val, format='%d/%m/%Y %H:%M')
    else:
        return pd.to_datetime(val, format='%b %d, %Y %I:%M %p')

df['txn_time_clean'] = df['txn_time'].apply(parse_txn_time)
print("Failed to parse:", df['txn_time_clean'].isna().sum())
df[['txn_time', 'txn_time_clean']].sample(10, random_state=1)

Failed to parse: 0


,txn_time,txn_time_clean
79900,2026-05-06 02:37:02,2026-05-06 02:37:02
57056,27/02/2025 13:26,2025-02-27 13:26:00
133219,2025-10-13 02:15:03,2025-10-13 02:15:03
42819,2026-01-07 21:25:57,2026-01-07 21:25:57
178284,2025-09-19 10:38:43,2025-09-19 10:38:43
87476,26/05/2025 11:06,2025-05-26 11:06:00
68436,2026-01-12 13:59:34,2026-01-12 13:59:34
155665,"May 06, 2026 11:08 AM",2026-05-06 11:08:00
61723,19/03/2026 22:04,2026-03-19 22:04:00
31418,30/03/2025 13:59,2025-03-30 13:59:00


 `msisdn` shall be used as the customer identifier for aggregation, since no dedicated 
`customer_id` exists. `msisdn` is the natural unit for mobile-money data (SIM = account), and 
is more reliable than `reg_id` (too many unique values, likely inflated by data-entry 
inconsistencies) or `account_name` (too few unique values, likely due to shared/generic 
names). This means the analysis operates at SIM level, not strictly at real-world-person 
level which is a limitation to flag given the multi-SIM issue found in the audit.

## RFM Features
# 1. Frequency
# 2. Recency
# 3. Monetary: Total Volume & Net Flow


# 1. Frequency

In [82]:
# The number of transactions per customer
freq = df.groupby('msisdn').size()
freq.head(10)

msisdn
254700000000      7
254700000003     39
254700000005      8
254700000007     17
254700000008      6
254700000011    108
254700000016     39
254700000021     36
254700000026     28
254700000028     23
dtype: int64

In [83]:
# To know how the transactions frequency values are distributed
freq.describe()

count    4825.000000
mean       38.051813
std        31.492086
min         1.000000
25%        15.000000
50%        29.000000
75%        52.000000
max       229.000000
dtype: float64

In [84]:
# Transaction Frequency Check according to the segment
freq_with_segment = df.groupby('msisdn')['segment'].first()
combined = pd.DataFrame({'freq': freq, 'segment': freq_with_segment})
combined.sort_values('freq', ascending=False).head(10)

,freq,segment
msisdn,,
254700005367,229,retail
254700013634,225,retail
254700000317,223,retail
254700004944,217,retail
254700003801,213,retail
254700013580,195,retail
254700001669,193,retail
254700012680,184,retail
254700001882,177,agent



**Outlier check.** Applying the IQR rule (Q1=15, Q3=52, IQR=37, upper fence=107.5) flags 
several high-frequency customers, up to a maximum of 229 transactions.

**Hypothesis tested:** high frequency is explained by `segment` (merchant/agent accounts 
naturally transacting more than retail). This hypothesis was however rejected. The top 10 highest-frequency customers are overwhelmingly labeled "retail," not merchant/agent segment and segment does not explain this pattern.

**Decision:** It is retained as a genuine signal instead of being treated as an error. No evidence of 
duplicate/bot-like transactions was found, and mobile-money retail users can legitimately 
transact at high volume. This follows the textbook's guidance against deleting behavioural 
extremes without a domain justification.

**Definition:** total count of transactions per customer, across all available history.

**Window:** full dataset history.

**Upstream fields:** `msisdn` (customer grouping key), `txn_id` (row-count basis).

**Write-time:** both fields are written at the moment each transaction occurs.

**Fairness note:** transaction frequency could correlate with income/employment type (e.g. 
salaried vs. informal workers transact differently). It is worth monitoring for indirect proxy 
effects, though no direct protected attribute is used.

In [85]:
#Frequency Variable saved and locked in the rfm table
rfm = pd.DataFrame({'msisdn': freq.index, 'frequency': freq.values})
rfm.head()

,msisdn,frequency
0,254700000000,7
1,254700000003,39
2,254700000005,8
3,254700000007,17
4,254700000008,6


# 2. Recency

In [86]:
# What the most recent transaction per customer?
last_txn = df.groupby('msisdn')['txn_time_clean'].max()
last_txn.head(10)

msisdn
254700000000   2026-07-06 05:28:00
254700000003   2026-07-13 08:02:00
254700000005   2026-06-05 23:30:00
254700000007   2026-07-26 06:54:00
254700000008   2025-12-16 16:55:50
254700000011   2026-07-31 00:56:00
254700000016   2026-07-08 23:31:00
254700000021   2026-07-30 05:37:00
254700000026   2026-07-08 07:14:00
254700000028   2026-07-20 15:11:00
Name: txn_time_clean, dtype: datetime64[us]

In [87]:
# How far is the most recent transaction to the latest dataset transaction? This helps us know how recently they were financially active.
reference_date = df['txn_time_clean'].max()
recency = (reference_date - last_txn).dt.days
recency.head(10)

msisdn
254700000000     25
254700000003     18
254700000005     55
254700000007      5
254700000008    227
254700000011      0
254700000016     22
254700000021      1
254700000026     23
254700000028     11
Name: txn_time_clean, dtype: int64

In [88]:
# Recency Feature saved and locked in the rfm table
rfm['recency'] = rfm['msisdn'].map(recency)
rfm.head()

,msisdn,frequency,recency
0,254700000000,7,25
1,254700000003,39,18
2,254700000005,8,55
3,254700000007,17,5
4,254700000008,6,227



**Definition:** number of days between each customer's most recent transaction and the most 
recent transaction date found anywhere in the dataset (used as the reference "now," since 
this is historical data, not a live system).

**Window:** full dataset history (no rolling window because it measures distance to a single 
reference point).

**Upstream fields:** `msisdn` (customer grouping key), `txn_time_clean` (cleaned timestamp).

**Write-time:** `txn_time_clean` is derived from `txn_time`, which is written at the moment 
each transaction occurs. There is no lag.

**Why predictive:** separates customers who are still actively using their account (low 
recency, more observable, more current information) from customers who have gone dormant 
(high recency, e.g. 227 days). This matters especially for thin-file borrowers, where 
mobile-money activity is the only available evidence of financial behaviour, unlike a 
traditional credit-bureau history.

**Fairness note:** Not directly tied to a protected attribute, but dormancy could correlate 
with rural/urban access to network infrastructure. It is worth noting if regional patterns emerge.

# 3. Monetary: Total Volume & Net Flow

In [89]:
# Money volume per customer: The amount of money that moved through a customer's account regardless of the type of transaction

total_volume = df.groupby('msisdn')['amount_clean'].apply(lambda x: x.abs().sum())
total_volume.head(10)

msisdn
254700000000     12950.0
254700000003     63583.0
254700000005     18902.0
254700000007     21390.0
254700000008      7639.0
254700000011    174085.0
254700000016     84566.0
254700000021     64607.0
254700000026     66300.0
254700000028     40260.0
Name: amount_clean, dtype: float64

In [90]:
#Net Flow: Overall, did the customer end up with more money coming in or coming out?

net_flow = df.groupby('msisdn')['amount_clean'].sum()
net_flow.head(10)

msisdn
254700000000    -4016.0
254700000003   -21175.0
254700000005    -8296.0
254700000007      322.0
254700000008    -5083.0
254700000011   -35109.0
254700000016     1340.0
254700000021   -11589.0
254700000026   -24390.0
254700000028    12414.0
Name: amount_clean, dtype: float64

In [91]:
#Monetary Feature saved and locked in the rfm table

rfm['total_volume'] = rfm['msisdn'].map(total_volume)
rfm['net_flow'] = rfm['msisdn'].map(net_flow)
rfm.head()

,msisdn,frequency,recency,total_volume,net_flow
0,254700000000,7,25,12950.0,-4016.0
1,254700000003,39,18,63583.0,-21175.0
2,254700000005,8,55,18902.0,-8296.0
3,254700000007,17,5,21390.0,322.0
4,254700000008,6,227,7639.0,-5083.0


### Total Volume

**Definition:** sum of the absolute value of every transaction amount per customer, capturing 
overall financial activity regardless of direction (deposits and withdrawals both count 
toward it, rather than cancelling out).

**Window:** full dataset history (no rolling window applied).

**Upstream fields:** `msisdn` (customer grouping key), `amount_clean` (cleaned transaction 
amount).

**Write-time:** `amount_clean` is derived from `amount`, written at the moment each 
transaction occurs. There is no lag.

**Why predictive:** deposits and withdrawals both signal an engaged and active account.A
direction-agnostic measure of how much financial activity flows through the customer's 
account overall.

**Fairness note:** higher-income customers naturally move larger absolute amounts. This 
feature should be interpreted alongside ratio-based features (Section: Ratios) to avoid 
penalizing lower-income but well-managed accounts.

### Net Flow

**Definition:** sum of the signed transaction amounts per customer, showing whether they end 
up net positive (more money in than out) or net negative (more money out than in) overall.

**Window:** full dataset history (no rolling window applied).

**Upstream fields:** `msisdn` (customer grouping key), `amount_clean` (cleaned transaction 
amount, sign-preserved).

**Write-time:** `amount_clean` is derived from `amount`, written at the moment each 
transaction occurs. There is no lag.

**Why predictive:** a meaningfully different signal from raw activity volume like a net positive 
flow may indicate saving behaviour and financial stability, while a strongly net-negative 
customer may be under financial strain, both relevant to repayment capacity.

**Fairness note:** net flow can be influenced by irregular income timing (e.g. informal 
workers paid in lump sums) rather than genuine financial mismanagement, this is worth caution 
against over-penalizing.

# Ratios
# 1. Outflow/Inflow Ratio & No Inflow Flag
# 2. Airtime Share of Spend

# 1. Outflow/Inflow Ratio & No Inflow Flag

In [92]:
# Keep only rows where money came IN (positive amounts = deposits/receipts),
# then sum those amounts per customer
inflow = df[df['amount_clean'] > 0].groupby('msisdn')['amount_clean'].sum()

# Keep only rows where money went OUT (negative amounts = withdrawals/payments),
# then flip the sign to positive and sum per customer
outflow = df[df['amount_clean'] < 0].groupby('msisdn')['amount_clean'].apply(lambda x: x.abs().sum())

In [93]:
# Divide each customer's total outflow by their total inflow
# A ratio > 1 means they spend more than they bring in; < 1 means they save more than they spend
outflow_inflow_ratio = outflow / inflow

# Show the first 10 customers' ratios to sanity-check the output
outflow_inflow_ratio.head(10)

msisdn
254700000000    1.899037
254700000003    1.998632
254700000005    2.564398
254700000007    0.970339
254700000008    4.977308
254700000011    1.505253
254700000016    0.968803
254700000021    1.437172
254700000026    2.163923
254700000028    0.528648
Name: amount_clean, dtype: float64

In [94]:
# Check the whole column for infinite or missing values, not just the first 10 rows
import numpy as np
print("Infinite values:", np.isinf(outflow_inflow_ratio).sum())
print("Missing values:", outflow_inflow_ratio.isna().sum())

Infinite values: 0
Missing values: 147


In [95]:
# Check: how many customers are in the full customer list vs. in the inflow-only and outflow-only groups
print("Total customers:", df['msisdn'].nunique())
print("Customers with at least one inflow:", inflow.shape[0])
print("Customers with at least one outflow:", outflow.shape[0])

Total customers: 4825
Customers with at least one inflow: 4694
Customers with at least one outflow: 4809


In [96]:
# Flag customers who have zero recorded inflow at all because this absence is itself meaningful,
# rather than something to disguise with an arbitrary filled-in ratio value
rfm['no_inflow_flag'] = (~rfm['msisdn'].isin(inflow.index)).astype(int)

# Attach the ratio as it is, NaN will naturally appear for the 131 customers with no inflow,
# and the model can be told to treat NaN + the flag together as their own distinct pattern
rfm['outflow_inflow_ratio'] = rfm['msisdn'].map(outflow_inflow_ratio)

rfm.head(10)

,msisdn,frequency,recency,total_volume,net_flow,no_inflow_flag,outflow_inflow_ratio
0,254700000000,7,25,12950.0,-4016.0,0,1.899037
1,254700000003,39,18,63583.0,-21175.0,0,1.998632
2,254700000005,8,55,18902.0,-8296.0,0,2.564398
3,254700000007,17,5,21390.0,322.0,0,0.970339
4,254700000008,6,227,7639.0,-5083.0,0,4.977308
5,254700000011,108,0,174085.0,-35109.0,0,1.505253
6,254700000016,39,22,84566.0,1340.0,0,0.968803
7,254700000021,36,1,64607.0,-11589.0,0,1.437172
8,254700000026,28,23,66300.0,-24390.0,0,2.163923
9,254700000028,23,11,40260.0,12414.0,0,0.528648


### Feature: Outflow/Inflow Ratio

**Definition:** ratio of total money withdrawn/spent to total money deposited, per customer 
which is a scale-free measure of spending relative to income, independent of the customer's absolute 
transaction volume.

**Window:** full dataset history (no rolling window applied).

**Upstream fields:** `msisdn` (customer grouping key), `amount_clean` (signed transaction 
amount).

**Write-time:** `amount_clean` is derived from `amount`, written at the moment each 
transaction occurs. There is no lag.

**Data quality note:** 131 customers (of 4,825) have zero recorded inflow, producing `NaN` 
for this ratio rather than a computable value. Rather than filling this with an arbitrary 
number, the missingness is preserved and captured separately via `no_inflow_flag`, since the 
absence of any recorded inflow is itself a meaningful pattern (e.g. income arriving outside 
the mobile-money system) which is consistent with the MNAR treatment applied elsewhere in this 
analysis.

**Why predictive:** unlike `total_volume` or `net_flow`, this ratio is scale-free, it can 
distinguish a high-income customer who overspends from a low-income customer who overspends, 
which raw monetary totals alone would conflate.

**Fairness note:** customers paid in irregular or informal channels (e.g. cash-based income) 
may appear as high-risk by this metric despite being financially stable which a proxy risk worth 
flagging to a credit officer rather than fully automating a decision on.

### Feature: No Inflow Flag

**Definition:** binary indicator (1/0) marking customers who have zero recorded deposit 
transactions in the dataset.

**Window:** full dataset history.

**Upstream fields:** `msisdn`, `amount_clean` (used to determine presence/absence of any 
positive-value transaction).

**Write-time:** derived from `amount_clean`, written at the moment each transaction occurs.

**Why predictive:** flags customers whose income source is entirely unobserved by this 
dataset which is a meaningfully different risk profile than someone with genuinely low income, since 
the model has no visibility into how this customer sustains their spending.

**Fairness note:** this flag could disproportionately capture customers in informal or 
cash-heavy economic segments — should be reviewed for proxy discrimination before being 
used in an automated decision, per Remark 2.3.

# 2. Airtime Share of Spend

In [97]:
# Total amount spent specifically on airtime, per customer
# (filtering to only airtime-type transactions before summing)
airtime_spend = df[df['txn_type'] == 'airtime'].groupby('msisdn')['amount_clean'].apply(lambda x: x.abs().sum())

# Airtime spend as a share of total outflow (spending) but not total_volume, since we want
# to compare airtime specifically against other spending, not against money coming in too
airtime_share = airtime_spend / outflow

# Fill customers with zero airtime transactions as 0% share (not NaN)
# unlike the inflow case, "no airtime spend" is a normal, valid zero, not a data gap
airtime_share = airtime_share.fillna(0)

rfm['airtime_share'] = rfm['msisdn'].map(airtime_share)
rfm[['msisdn', 'airtime_share']].head(10)

,msisdn,airtime_share
0,254700000000,0.000000
1,254700000003,0.014276
2,254700000005,0.000000
3,254700000007,0.030473
4,254700000008,0.020123
5,254700000011,0.008958
6,254700000016,0.005071
7,254700000021,0.008557
8,254700000026,0.012262
9,254700000028,0.002370


In [98]:
# The rfm table including the airtime share
rfm.head(10)

,msisdn,frequency,recency,total_volume,net_flow,no_inflow_flag,outflow_inflow_ratio,airtime_share
0,254700000000,7,25,12950.0,-4016.0,0,1.899037,0.000000
1,254700000003,39,18,63583.0,-21175.0,0,1.998632,0.014276
2,254700000005,8,55,18902.0,-8296.0,0,2.564398,0.000000
3,254700000007,17,5,21390.0,322.0,0,0.970339,0.030473
4,254700000008,6,227,7639.0,-5083.0,0,4.977308,0.020123
5,254700000011,108,0,174085.0,-35109.0,0,1.505253,0.008958
6,254700000016,39,22,84566.0,1340.0,0,0.968803,0.005071
7,254700000021,36,1,64607.0,-11589.0,0,1.437172,0.008557
8,254700000026,28,23,66300.0,-24390.0,0,2.163923,0.012262
9,254700000028,23,11,40260.0,12414.0,0,0.528648,0.002370


### Feature: Airtime Share of Spend

**Definition:** proportion of a customer's total spending (outflow) that goes specifically 
toward airtime purchases.

**Window:** full dataset history (no rolling window applied).

**Upstream fields:** `msisdn`, `txn_type` (to isolate airtime transactions), `amount_clean` 
(transaction value).

**Write-time:** `txn_type` and `amount_clean` are both written at the moment each transaction 
occurs, There is no lag.

**Data quality note:** customers with no airtime transactions are filled with 0, distinct 
from the earlier `no_inflow_flag` case. Here, a zero genuinely means "no airtime spend 
observed," not a hidden data gap, so no separate missingness flag is needed.

**Why predictive:** airtime spend is a routine, small, recurring cost,  a consistently 
non-zero share may indicate a stable, active phone/communication pattern, while sudden 
absence could indicate financial strain or account inactivity.

**Fairness note:** airtime spend patterns could vary by age or income bracket (e.g. younger, 
lower-income users often rely more heavily on prepaid airtime) which is worthy reviewing as a 
potential indirect proxy per Remark 2.3.

# Velocity / Trend
# 1. Balance Trend
# 2. Transaction Count Trend
# 3. Amount trend

# 1. Balance Trend

In [99]:
# Sort all transactions by time, so "first" and "last" per customer are accurate
df_sorted = df.sort_values('txn_time_clean')

In [100]:
# For each customer, get their balance at their earliest transaction
first_balance = df_sorted.groupby('msisdn')['balance_after'].first()

# For each customer, get their balance at their most recent transaction
last_balance = df_sorted.groupby('msisdn')['balance_after'].last()

In [101]:
# Positive = balance grew over the observed period; negative = balance shrank
balance_trend = last_balance - first_balance
balance_trend.head(10)

msisdn
254700000000    -304.0
254700000003    4919.0
254700000005    1557.0
254700000007   -3449.0
254700000008   -1638.0
254700000011    4609.0
254700000016   -7055.0
254700000021   -1603.0
254700000026    6255.0
254700000028   -4803.0
Name: balance_after, dtype: float64

In [102]:
# Save the feature and lock it in
rfm['balance_trend'] = rfm['msisdn'].map(balance_trend)
rfm.head(10)

,msisdn,frequency,recency,total_volume,net_flow,no_inflow_flag,outflow_inflow_ratio,airtime_share,balance_trend
0,254700000000,7,25,12950.0,-4016.0,0,1.899037,0.000000,-304.0
1,254700000003,39,18,63583.0,-21175.0,0,1.998632,0.014276,4919.0
2,254700000005,8,55,18902.0,-8296.0,0,2.564398,0.000000,1557.0
3,254700000007,17,5,21390.0,322.0,0,0.970339,0.030473,-3449.0
4,254700000008,6,227,7639.0,-5083.0,0,4.977308,0.020123,-1638.0
5,254700000011,108,0,174085.0,-35109.0,0,1.505253,0.008958,4609.0
6,254700000016,39,22,84566.0,1340.0,0,0.968803,0.005071,-7055.0
7,254700000021,36,1,64607.0,-11589.0,0,1.437172,0.008557,-1603.0
8,254700000026,28,23,66300.0,-24390.0,0,2.163923,0.012262,6255.0
9,254700000028,23,11,40260.0,12414.0,0,0.528648,0.002370,-4803.0


### Feature: Balance Trend

**Definition:** Balance Trend is a change in account balance from a customer's first recorded transaction to their most recent transaction. It is a simple directional measure of whether their balance is 
growing or shrinking over the observed period.

**Window:** full observed history per customer (first transaction to last transaction — this 
window naturally varies in length between customers, since some have longer histories than 
others).

**Upstream fields:** `msisdn`, `txn_time_clean` (to establish chronological order), 
`balance_after` (the balance snapshot at each transaction).

**Write-time:** `txn_time_clean` and `balance_after` are both written at the moment each 
transaction occurs — no lag.

**Why predictive:** a customer whose balance is trending upward shows signs of accumulating 
savings and financial stability, while a declining trend may indicate financial stress which is
directly relevant to repayment capacity, and captures a *direction* that a single balance 
snapshot alone cannot.

**Limitation to note:** this measures only the two endpoints (first vs last), not the shape 
of the trend in between. A customer could have had a large dip and recovery in the middle 
and still show a flat or positive trend here. A more robust version would fit a slope across 
all balance points, not just first/last.

**Fairness note:** balance trend can be affected by one-off life events (medical expenses, 
school fees) unrelated to creditworthiness, worth interpreting alongside other stability 
features rather than in isolation.

# 2. Transaction Count Trend

In [103]:
# Reference point = latest date in the whole dataset
ref_date = df['txn_time_clean'].max()

# Count each customer's transactions in just the last 30 days
recent_txns = df[df['txn_time_clean'] > ref_date - pd.Timedelta(days=30)]
recent_count = recent_txns.groupby('msisdn').size()

# Each customer's "typical" transactions-per-30-days, based on their full history
total_days_active = (df.groupby('msisdn')['txn_time_clean'].max() 
                      - df.groupby('msisdn')['txn_time_clean'].min()).dt.days.clip(lower=1)
overall_rate_per_30d = (freq / total_days_active) * 30

# Trend = recent pace minus typical pace (positive = accelerating, negative = slowing down)
count_trend = rfm['msisdn'].map(recent_count).fillna(0) - rfm['msisdn'].map(overall_rate_per_30d)
rfm['txn_count_trend'] = count_trend
rfm[['msisdn', 'txn_count_trend']].head(10)

,msisdn,txn_count_trend
0,254700000000,0.481481
1,254700000003,-1.245681
2,254700000005,-0.522876
3,254700000007,0.044944
4,254700000008,-0.580645
5,254700000011,-2.124764
6,254700000016,-0.271845
7,254700000021,-1.204082
8,254700000026,-0.631068
9,254700000028,0.644401


In [104]:
# Look up each customer's "typical" 30-day transaction rate (calculated earlier),
# so we can compare it side-by-side against their txn_count_trend value
# this tells us whether a trend like -1.25 is a big slowdown or a tiny one,
# relative to that specific customer's normal pace
rfm['msisdn'].map(overall_rate_per_30d).head(10)

0    0.518519
1    2.245681
2    0.522876
3    0.955056
4    0.580645
5    6.124764
6    2.271845
7    2.204082
8    1.631068
9    1.355599
Name: msisdn, dtype: float64

In [105]:
# Lock the transaction count trend into the rfm table
rfm['txn_count_trend'] = count_trend

rfm.head(10)

,msisdn,frequency,recency,total_volume,net_flow,no_inflow_flag,outflow_inflow_ratio,airtime_share,balance_trend,txn_count_trend
0,254700000000,7,25,12950.0,-4016.0,0,1.899037,0.000000,-304.0,0.481481
1,254700000003,39,18,63583.0,-21175.0,0,1.998632,0.014276,4919.0,-1.245681
2,254700000005,8,55,18902.0,-8296.0,0,2.564398,0.000000,1557.0,-0.522876
3,254700000007,17,5,21390.0,322.0,0,0.970339,0.030473,-3449.0,0.044944
4,254700000008,6,227,7639.0,-5083.0,0,4.977308,0.020123,-1638.0,-0.580645
5,254700000011,108,0,174085.0,-35109.0,0,1.505253,0.008958,4609.0,-2.124764
6,254700000016,39,22,84566.0,1340.0,0,0.968803,0.005071,-7055.0,-0.271845
7,254700000021,36,1,64607.0,-11589.0,0,1.437172,0.008557,-1603.0,-1.204082
8,254700000026,28,23,66300.0,-24390.0,0,2.163923,0.012262,6255.0,-0.631068
9,254700000028,23,11,40260.0,12414.0,0,0.528648,0.002370,-4803.0,0.644401


### Feature: Transaction Count Trend (30-day)

**Definition:** difference between a customer's transaction count in the most recent 30 days 
and their typical 30-day rate across their full observed history. Positive means recently 
accelerating, negative means recently slowing down.

**Window:** last 30 days, compared against full dataset history.

**Upstream fields:** `msisdn`, `txn_time_clean`.

**Write-time:** `txn_time_clean` derived from `txn_time`, written at the moment each 
transaction occurs.

**Why predictive:** captures direction of engagement, not just current activity level, a 
customer slowing down may signal financial distress or disengagement, even if their overall 
frequency still looks healthy.

**Fairness note:** short-term slowdowns could reflect temporary circumstances (illness, 
travel, seasonal informal work) unrelated to creditworthiness and should be interpreted 
alongside longer-window features, not in isolation.

# 3. Amount Trend

In [106]:
# Average transaction size in the last 30 days, per customer
recent_avg_amt = recent_txns.groupby('msisdn')['amount_clean'].apply(lambda x: x.abs().mean())

# Average transaction size across their full history, per customer
overall_avg_amt = df.groupby('msisdn')['amount_clean'].apply(lambda x: x.abs().mean())

# Trend = recent average minus overall average (positive = transacting bigger amounts lately)
amount_trend = rfm['msisdn'].map(recent_avg_amt).fillna(0) - rfm['msisdn'].map(overall_avg_amt)

In [107]:
# Lock the amount trend into the rfm table
rfm['amount_trend'] = amount_trend

# View the full feature table so far
rfm.head(10)

,msisdn,frequency,recency,total_volume,net_flow,no_inflow_flag,outflow_inflow_ratio,airtime_share,balance_trend,txn_count_trend,amount_trend
0,254700000000,7,25,12950.0,-4016.0,0,1.899037,0.000000,-304.0,0.481481,-659.000000
1,254700000003,39,18,63583.0,-21175.0,0,1.998632,0.014276,4919.0,-1.245681,396.666667
2,254700000005,8,55,18902.0,-8296.0,0,2.564398,0.000000,1557.0,-0.522876,-2362.750000
3,254700000007,17,5,21390.0,322.0,0,0.970339,0.030473,-3449.0,0.044944,-992.235294
4,254700000008,6,227,7639.0,-5083.0,0,4.977308,0.020123,-1638.0,-0.580645,-1273.166667
5,254700000011,108,0,174085.0,-35109.0,0,1.505253,0.008958,4609.0,-2.124764,-27.898148
6,254700000016,39,22,84566.0,1340.0,0,0.968803,0.005071,-7055.0,-0.271845,-1443.358974
7,254700000021,36,1,64607.0,-11589.0,0,1.437172,0.008557,-1603.0,-1.204082,169.361111
8,254700000026,28,23,66300.0,-24390.0,0,2.163923,0.012262,6255.0,-0.631068,4244.142857
9,254700000028,23,11,40260.0,12414.0,0,0.528648,0.002370,-4803.0,0.644401,704.565217


### Feature: Amount Trend (30-day)

**Definition:** difference between a customer's average transaction size in the most recent 
30 days and their average transaction size across their full history. Positive means 
transacting in larger amounts recently, negative means smaller.

**Window:** last 30 days, compared against full dataset history.

**Upstream fields:** `msisdn`, `txn_time_clean`, `amount_clean`.

**Write-time:** both derived fields are written at the moment each transaction occurs, There is no lag.

**Why predictive:** a rising trend could indicate growing financial capacity or a large 
one-off event; a shrinking trend could indicate reduced income or spending caution, either 
way, a different signal than the flat `total_volume` figure alone.

**Fairness note:** large one-off transactions (e.g. medical bills, school fees) can distort 
this trend for a single 30-day window, it is best interpreted alongside other stability features, 
not as a standalone risk signal.

# Stability
# 1. Weekly Transaction Count Stability
# 2. Distinct Counterparties
# 3. Transaction Amount std

# 1. Weekly Transaction Count Stability

In [108]:
# Give every transaction a "week" label based on its timestamp
# (multiple transactions in the same 7-day period get grouped as one week)
df['week'] = df['txn_time_clean'].dt.to_period('W')

In [109]:
df[['txn_time_clean', 'week']].head(5)

,txn_time_clean,week
0,2026-06-30 14:24:19,2026-06-29/2026-07-05
1,2026-01-29 13:17:00,2026-01-26/2026-02-01
2,2025-09-16 00:55:00,2025-09-15/2025-09-21
3,2026-04-07 11:21:00,2026-04-06/2026-04-12
4,2025-12-20 11:05:00,2025-12-15/2025-12-21


In [110]:
# For each customer, in each week, count how many transactions they made
# This gives one row per (customer, week) combination, with a count
weekly_counts = df.groupby(['msisdn', 'week']).size()
weekly_counts.head(20)

msisdn        week                 
254700000000  2025-05-26/2025-06-01    1
              2025-11-03/2025-11-09    1
              2025-11-17/2025-11-23    1
              2026-02-09/2026-02-15    1
              2026-03-30/2026-04-05    1
              2026-05-04/2026-05-10    1
              2026-07-06/2026-07-12    1
254700000003  2025-02-03/2025-02-09    2
              2025-03-03/2025-03-09    2
              2025-03-10/2025-03-16    1
              2025-03-31/2025-04-06    1
              2025-04-14/2025-04-20    2
              2025-04-28/2025-05-04    1
              2025-05-12/2025-05-18    1
              2025-06-02/2025-06-08    1
              2025-06-23/2025-06-29    1
              2025-07-07/2025-07-13    1
              2025-07-14/2025-07-20    1
              2025-09-15/2025-09-21    1
              2025-11-10/2025-11-16    1
dtype: int64

In [111]:
# For each customer, how do their weekly transaction count vary?
# High number = wildly inconsistent activity
# Low number = steady, predictable activity
weekly_count_std = weekly_counts.groupby('msisdn').std()
weekly_count_std.head(10)

msisdn
254700000000    0.000000
254700000003    0.628890
254700000005    0.377964
254700000007    0.351866
254700000008    0.000000
254700000011    1.262856
254700000016    0.768852
254700000021    0.510964
254700000026    0.481543
254700000028    0.418854
dtype: float64

In [112]:
print("Missing values:", weekly_count_std.isna().sum())

Missing values: 35


In [113]:
# Check: do the NaN customers have very few active weeks?
weeks_per_customer = weekly_counts.groupby('msisdn').size()
weeks_per_customer[weekly_count_std.isna()].describe()

count    35.0
mean      1.0
std       0.0
min       1.0
25%       1.0
50%       1.0
75%       1.0
max       1.0
dtype: float64

In [114]:
# Flag customers with only 1 active week; not enough history to judge stability from,
# distinct from customers who were genuinely erratic across many weeks
rfm['insufficient_history_flag'] = rfm['msisdn'].map(weeks_per_customer).eq(1).astype(int)

# Fill their std with 0 (treated as "assume stable" due to lack of evidence),
# while genuinely erratic customers keep their real calculated value
rfm['weekly_count_std'] = rfm['msisdn'].map(weekly_count_std).fillna(0)

rfm.head(10)

,msisdn,frequency,recency,total_volume,net_flow,no_inflow_flag,outflow_inflow_ratio,airtime_share,balance_trend,txn_count_trend,amount_trend,insufficient_history_flag,weekly_count_std
0,254700000000,7,25,12950.0,-4016.0,0,1.899037,0.000000,-304.0,0.481481,-659.000000,0,0.000000
1,254700000003,39,18,63583.0,-21175.0,0,1.998632,0.014276,4919.0,-1.245681,396.666667,0,0.628890
2,254700000005,8,55,18902.0,-8296.0,0,2.564398,0.000000,1557.0,-0.522876,-2362.750000,0,0.377964
3,254700000007,17,5,21390.0,322.0,0,0.970339,0.030473,-3449.0,0.044944,-992.235294,0,0.351866
4,254700000008,6,227,7639.0,-5083.0,0,4.977308,0.020123,-1638.0,-0.580645,-1273.166667,0,0.000000
5,254700000011,108,0,174085.0,-35109.0,0,1.505253,0.008958,4609.0,-2.124764,-27.898148,0,1.262856
6,254700000016,39,22,84566.0,1340.0,0,0.968803,0.005071,-7055.0,-0.271845,-1443.358974,0,0.768852
7,254700000021,36,1,64607.0,-11589.0,0,1.437172,0.008557,-1603.0,-1.204082,169.361111,0,0.510964
8,254700000026,28,23,66300.0,-24390.0,0,2.163923,0.012262,6255.0,-0.631068,4244.142857,0,0.481543
9,254700000028,23,11,40260.0,12414.0,0,0.528648,0.002370,-4803.0,0.644401,704.565217,0,0.418854


### Feature: Weekly Transaction Count Stability

**Definition:** standard deviation of a customer's transaction count across the weeks they 
were active where low values indicate steady, predictable activity (e.g. salaried-style 
patterns); high values indicate irregular, bursty activity (e.g. informal/entrepreneurial 
income patterns).

**Window:** full dataset history, aggregated to weekly buckets.

**Upstream fields:** `msisdn`, `txn_time_clean` (used to derive the `week` grouping).

**Write-time:** `txn_time_clean` is derived from `txn_time`, written at the moment each 
transaction occurs. There is no lag.

**Data quality note:** 35 customers have only 1 active week in their entire history, making 
standard deviation mathematically undefined (cannot measure spread from a single point). 
These are filled with 0 and separately marked via `insufficient_history_flag`, distinguishing 
"assumed stable due to limited history" from "genuinely verified as stable."

**Why predictive:** captures income/activity regularity, a concept closely tied to repayment 
predictability. A customer with wildly unpredictable activity is harder to assess than one 
with a steady, legible pattern, independent of how much total activity (Frequency) they have.

**Fairness note:** irregular activity is strongly associated with informal or entrepreneurial 
income sources rather than financial irresponsibility. This feature should be treated as a 
predictability signal, not a value judgment, and reviewed for proxy bias against informal-
sector workers.

# 2. Weekly Counterpaties

In [115]:
# Count how many different counterparties (people/agents) each customer transacted with
distinct_counterparties = df.groupby('msisdn')['counterparty'].nunique()

# Lock it into the rfm table
rfm['distinct_counterparties'] = rfm['msisdn'].map(distinct_counterparties)
rfm.head(10)

,msisdn,frequency,recency,total_volume,net_flow,no_inflow_flag,outflow_inflow_ratio,airtime_share,balance_trend,txn_count_trend,amount_trend,insufficient_history_flag,weekly_count_std,distinct_counterparties
0,254700000000,7,25,12950.0,-4016.0,0,1.899037,0.000000,-304.0,0.481481,-659.000000,0,0.000000,7
1,254700000003,39,18,63583.0,-21175.0,0,1.998632,0.014276,4919.0,-1.245681,396.666667,0,0.628890,39
2,254700000005,8,55,18902.0,-8296.0,0,2.564398,0.000000,1557.0,-0.522876,-2362.750000,0,0.377964,7
3,254700000007,17,5,21390.0,322.0,0,0.970339,0.030473,-3449.0,0.044944,-992.235294,0,0.351866,16
4,254700000008,6,227,7639.0,-5083.0,0,4.977308,0.020123,-1638.0,-0.580645,-1273.166667,0,0.000000,6
5,254700000011,108,0,174085.0,-35109.0,0,1.505253,0.008958,4609.0,-2.124764,-27.898148,0,1.262856,107
6,254700000016,39,22,84566.0,1340.0,0,0.968803,0.005071,-7055.0,-0.271845,-1443.358974,0,0.768852,38
7,254700000021,36,1,64607.0,-11589.0,0,1.437172,0.008557,-1603.0,-1.204082,169.361111,0,0.510964,36
8,254700000026,28,23,66300.0,-24390.0,0,2.163923,0.012262,6255.0,-0.631068,4244.142857,0,0.481543,28
9,254700000028,23,11,40260.0,12414.0,0,0.528648,0.002370,-4803.0,0.644401,704.565217,0,0.418854,22


### Feature: Distinct Counterparties

**Definition:** count of unique people/agents/merchants each customer transacted with across 
their full history. This measures the breadth of a customer's financial relationships.

**Window:** full dataset history.

**Upstream fields:** `msisdn`, `counterparty`.

**Write-time:** `counterparty` is written at the moment each transaction occurs, There is no lag.

**Why predictive:** a customer transacting with many different counterparties may have a 
broader, more diversified financial life (multiple income sources, varied spending), while 
one repeatedly transacting with the same few counterparties may have a narrower, more 
predictable or more dependent financial relationship structure.

**Fairness note:** counterparty diversity can reflect geographic or social network 
differences (e.g. rural customers may have fewer distinct agents nearby) rather than 
financial behaviour. This is worth caution as an indirect proxy for location/access.

# 3. Transaction Amount std

In [116]:
# For each customer, measure how consistent (or varied) their individual transaction sizes are
# Low std = steady, similar-sized transactions; high std = wildly varying amounts
amount_std = df.groupby('msisdn')['amount_clean'].apply(lambda x: x.abs().std())

# Lock it into the rfm table
rfm['amount_std'] = rfm['msisdn'].map(amount_std)
rfm.head(10)

,msisdn,frequency,recency,total_volume,net_flow,no_inflow_flag,outflow_inflow_ratio,airtime_share,balance_trend,txn_count_trend,amount_trend,insufficient_history_flag,weekly_count_std,distinct_counterparties,amount_std
0,254700000000,7,25,12950.0,-4016.0,0,1.899037,0.000000,-304.0,0.481481,-659.000000,0,0.000000,7,1045.925746
1,254700000003,39,18,63583.0,-21175.0,0,1.998632,0.014276,4919.0,-1.245681,396.666667,0,0.628890,39,1206.192625
2,254700000005,8,55,18902.0,-8296.0,0,2.564398,0.000000,1557.0,-0.522876,-2362.750000,0,0.377964,7,2303.380606
3,254700000007,17,5,21390.0,322.0,0,0.970339,0.030473,-3449.0,0.044944,-992.235294,0,0.351866,16,980.868335
4,254700000008,6,227,7639.0,-5083.0,0,4.977308,0.020123,-1638.0,-0.580645,-1273.166667,0,0.000000,6,1034.796872
5,254700000011,108,0,174085.0,-35109.0,0,1.505253,0.008958,4609.0,-2.124764,-27.898148,0,1.262856,107,1368.421985
6,254700000016,39,22,84566.0,1340.0,0,0.968803,0.005071,-7055.0,-0.271845,-1443.358974,0,0.768852,38,1842.113410
7,254700000021,36,1,64607.0,-11589.0,0,1.437172,0.008557,-1603.0,-1.204082,169.361111,0,0.510964,36,1425.034298
8,254700000026,28,23,66300.0,-24390.0,0,2.163923,0.012262,6255.0,-0.631068,4244.142857,0,0.481543,28,2007.735756
9,254700000028,23,11,40260.0,12414.0,0,0.528648,0.002370,-4803.0,0.644401,704.565217,0,0.418854,22,1054.197016


# Behavioural Flags
# 1. Night - Time Transaction Share

In [117]:
# Extract the hour of day from each transaction's timestamp
df['hour'] = df['txn_time_clean'].dt.hour

# Flag transactions happening late at night (e.g. 12am–5am)
df['is_night_txn'] = df['hour'].between(0, 5)

# Calculate what share of each customer's transactions happen at night
night_share = df.groupby('msisdn')['is_night_txn'].mean()

# Lock it into the rfm table
rfm['night_txn_share'] = rfm['msisdn'].map(night_share)
rfm.head(10)

,msisdn,frequency,recency,total_volume,net_flow,no_inflow_flag,outflow_inflow_ratio,airtime_share,balance_trend,txn_count_trend,amount_trend,insufficient_history_flag,weekly_count_std,distinct_counterparties,amount_std,night_txn_share
0,254700000000,7,25,12950.0,-4016.0,0,1.899037,0.000000,-304.0,0.481481,-659.000000,0,0.000000,7,1045.925746,0.285714
1,254700000003,39,18,63583.0,-21175.0,0,1.998632,0.014276,4919.0,-1.245681,396.666667,0,0.628890,39,1206.192625,0.179487
2,254700000005,8,55,18902.0,-8296.0,0,2.564398,0.000000,1557.0,-0.522876,-2362.750000,0,0.377964,7,2303.380606,0.375000
3,254700000007,17,5,21390.0,322.0,0,0.970339,0.030473,-3449.0,0.044944,-992.235294,0,0.351866,16,980.868335,0.176471
4,254700000008,6,227,7639.0,-5083.0,0,4.977308,0.020123,-1638.0,-0.580645,-1273.166667,0,0.000000,6,1034.796872,0.333333
5,254700000011,108,0,174085.0,-35109.0,0,1.505253,0.008958,4609.0,-2.124764,-27.898148,0,1.262856,107,1368.421985,0.268519
6,254700000016,39,22,84566.0,1340.0,0,0.968803,0.005071,-7055.0,-0.271845,-1443.358974,0,0.768852,38,1842.113410,0.153846
7,254700000021,36,1,64607.0,-11589.0,0,1.437172,0.008557,-1603.0,-1.204082,169.361111,0,0.510964,36,1425.034298,0.305556
8,254700000026,28,23,66300.0,-24390.0,0,2.163923,0.012262,6255.0,-0.631068,4244.142857,0,0.481543,28,2007.735756,0.071429
9,254700000028,23,11,40260.0,12414.0,0,0.528648,0.002370,-4803.0,0.644401,704.565217,0,0.418854,22,1054.197016,0.130435



### Feature: Night-time Transaction Share

**Definition:** proportion of a customer's transactions that occur between midnight and 5am. This is 
a behavioural pattern flag, distinct from the purely financial features above.

**Window:** full dataset history.

**Upstream fields:** `msisdn`, `txn_time_clean` (to extract the hour of each transaction).

**Write-time:** `txn_time_clean` derived from `txn_time`, written at the moment each 
transaction occurs — no lag.

**Why predictive:** the textbook (Table 2.2) explicitly identifies night-time transaction 
share as a behavioural risk-culture signal in mobile-money credit scoring; unusual 
transaction timing patterns can correlate with financial distress, informal/irregular 
work schedules, or atypical account usage.

**Fairness note (Remark 2.3):** this feature carries a real proxy-discrimination risk like
night-shift workers, certain occupations, or specific demographic/regional groups may 
transact at night for entirely legitimate reasons unrelated to creditworthiness. Per the 
textbook's guidance, this feature should be reviewed against protected attributes before 
being used in an automated decision, and treated as a flag for human review rather than a 
direct penalty.

# PART III: LEAKAGE ASSESSMENT MEMO


In [118]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, GroupKFold

# A small, fast feature set for this leakage test (not the full model)
base_features_num = ['amount_clean', 'balance_after']
base_features_cat = ['txn_type', 'segment', 'region']

# WITH the suspected leaky columns included
leaky_num = base_features_num + ['manual_review_score']
leaky_cat = base_features_cat + ['settlement_status']

In [119]:
# Build and Test the model with leakage

num_pipe = Pipeline([('impute', SimpleImputer(strategy='median')), ('scale', StandardScaler())])
cat_pipe = Pipeline([('impute', SimpleImputer(strategy='constant', fill_value='UNK')),
                      ('onehot', OneHotEncoder(handle_unknown='ignore'))])

prep_leaky = ColumnTransformer([('num', num_pipe, leaky_num), ('cat', cat_pipe, leaky_cat)])
model_leaky = Pipeline([('prep', prep_leaky), ('clf', LogisticRegression(max_iter=1000))])

scores_leaky = cross_val_score(model_leaky, df, df['is_fraud'],
                                 cv=GroupKFold(n_splits=5), groups=df['msisdn'],
                                 scoring='roc_auc')
print("WITH leaky columns - AUC:", scores_leaky.mean())

WITH leaky columns - AUC: 1.0


In [120]:
# Build and test the model without leakage
prep_clean = ColumnTransformer([('num', num_pipe, base_features_num), ('cat', cat_pipe, base_features_cat)])
model_clean = Pipeline([('prep', prep_clean), ('clf', LogisticRegression(max_iter=1000))])

scores_clean = cross_val_score(model_clean, df, df['is_fraud'],
                                 cv=GroupKFold(n_splits=5), groups=df['msisdn'],
                                 scoring='roc_auc')
print("WITHOUT leaky columns - AUC:", scores_clean.mean())

WITHOUT leaky columns - AUC: 0.5911344130307904


# Leakage Assessment

**Leaks identified:** `manual_review_score` and `settlement_status`.

**Mechanism  for `settlement_status`:** this column contains values such as "reversed," 
"pending," and "held". These states are only determined *after* a transaction has been 
processed and settled by the system. This is Cause 6 from Section 2.7.2, leakage from the 
data-collection process, where a field is written downstream of the outcome it is meant to 
predict. A transaction cannot be scored using its own future settlement outcome; this field 
would not exist in that form at the moment a real prediction is needed.

**Mechanism for `manual_review_score`:** this is a score generated by a human reviewer, which 
by definition can only be assigned *after* a transaction has already been flagged and 
examined, likely in direct response to suspected fraud. This is also Cause 6 (data-collection 
leakage): the review process itself is triggered by the very outcome (`is_fraud`) the model 
is meant to predict, making this feature a near-direct proxy for the label rather than a 
genuine predictive signal.

**Quantified impact:** cross-validated ROC-AUC (5-fold, grouped by `msisdn` to prevent 
customer-level leakage across folds) was **1.0** when both columns were included, versus 
**0.591** once removed — an inflation of roughly **0.41 AUC points**. An AUC of exactly 1.0 
is itself a red flag per the textbook's "too-good-to-be-true" detection screen (Section 
2.7.4).This indicates the model is reading the label directly off these two fields rather 
than learning genuine behavioural patterns.

**Recommended process change:** enforce a **feature lineage review** before any field is 
added to a training pipeline, specifically, documenting *when* each field is written 
relative to the event being predicted (its "write-time"), as done throughout this Feature 
Dictionary. Any field written by a downstream process (collections, manual review, settlement 
systems) triggered *by* the outcome should be excluded by default unless proven otherwise. 
This should be a mandatory sign-off step in the model development workflow, not an ad-hoc 
check left to individual analysts.

# Part IV: Reproducible Pipeline

In [ ]:
# Merging the engineered rfm features onto each transaction, matched by customer (msisdn)
df_final = df.merge(rfm, on='msisdn', how='left')

In [122]:
# Building the final clean pipeline

final_num_features = ['amount_clean', 'balance_after', 'frequency', 'recency', 
                       'total_volume', 'net_flow', 'outflow_inflow_ratio', 
                       'weekly_count_std', 'night_txn_share']
final_cat_features = ['txn_type', 'segment', 'region']

num_pipe_final = Pipeline([('impute', SimpleImputer(strategy='median')), ('scale', StandardScaler())])
cat_pipe_final = Pipeline([('impute', SimpleImputer(strategy='constant', fill_value='UNK')),
                            ('onehot', OneHotEncoder(handle_unknown='ignore'))])

prep_final = ColumnTransformer([('num', num_pipe_final, final_num_features),
                                  ('cat', cat_pipe_final, final_cat_features)])

final_pipeline = Pipeline([('prep', prep_final), ('clf', LogisticRegression(max_iter=1000))])

In [123]:
# Evaluation with group-aware cross-validation and report of the final AUC
final_scores = cross_val_score(final_pipeline, df_final, df_final['is_fraud'],
                                 cv=GroupKFold(n_splits=5), groups=df_final['msisdn'],
                                 scoring='roc_auc')
print("Final cross-validated ROC-AUC:", final_scores.mean())
print("Fold scores:", final_scores)

Final cross-validated ROC-AUC: 0.5942042070861522
Fold scores: [0.59085705 0.59749178 0.59596565 0.59378384 0.59292271]


In [124]:
import joblib

# Fit the pipeline on the FULL dataset (cross_val_score only evaluates, doesn't save a fitted model)
final_pipeline.fit(df_final, df_final['is_fraud'])

# Save the fitted pipeline to a file
joblib.dump(final_pipeline, '../src/final_pipeline.joblib')
print("Pipeline saved.")

Pipeline saved.


In [125]:
# Reload the pipeline fresh, as if you were the marker opening this file independently
loaded_pipeline = joblib.load('../src/final_pipeline.joblib')

# Test it can still make predictions
test_predictions = loaded_pipeline.predict_proba(df_final.head(5))
print(test_predictions)

[[0.90617898 0.09382102]
 [0.8907074  0.1092926 ]
 [0.93908862 0.06091138]
 [0.9037428  0.0962572 ]
 [0.86254097 0.13745903]]


## Reproducible Machine Learning Pipeline

A single scikit-learn `Pipeline` was built combining a `ColumnTransformer` (median imputation 
+ standard scaling for numeric features, constant imputation + one-hot encoding for 
categorical features) with a `LogisticRegression` classifier. All preprocessing steps are 
contained *within* the pipeline object itself, so cross-validation re-fits every 
transformation on each fold's training data only — no preprocessing occurs outside this 
structure, avoiding the leakage patterns identified in Part 3.

**Features used:** engineered behavioural features (`frequency`, `recency`, `total_volume`, 
`net_flow`, `outflow_inflow_ratio`, `weekly_count_std`, `night_txn_share`) alongside raw 
transaction-level features (`amount_clean`, `balance_after`, `txn_type`, `segment`, 
`region`) — explicitly excluding the two leaky columns (`manual_review_score`, 
`settlement_status`) identified and removed in Part 3.

**Evaluation:** 5-fold `GroupKFold` cross-validation, grouped by `msisdn`, ensuring no 
customer's transactions appear in both the training and validation portion of any fold.

**Final cross-validated ROC-AUC: 0.594** (fold range: 0.591–0.597), consistent with the honest 
"post-leakage-removal" baseline established in Part 3, confirming this model reflects genuine 
predictive signal rather than leaked information.

The fitted pipeline was serialised with `joblib` (`final_pipeline.joblib`) and independently 
reloaded to confirm it produces valid predictions without modification, satisfying the 
reproducibility requirement.